---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

---

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
from datetime import datetime, timedelta
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

In [ ]:
#A Menyiapkan Streaming Source
folder_stream = "tugas10_stream"
folder_output = "tugas10_output"
folder_ckpt = "tugas10_checkpoint"
for f in [folder_stream, folder_output, folder_ckpt]:
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Skema:", skema_pesanan.simpleString())
print("df_stream.isStreaming:", df_stream.isStreaming)

In [ ]:
#B. Agregasi per Cabang
df_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang = df_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi_cabang dimulai.")

In [ ]:
#C. Agregasi Berbasis Window Waktu
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_harga")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi_waktu dimulai.")

In [ ]:
#D. Menguji dengan Data Simulasi 
kolom = ["pesanan_id", "nama_cabang", "nama_menu", "qty", "harga_satuan", "waktu_pesan"]
semua_data = []
sekarang = datetime.now()

def t(menit_lalu):
    return (sekarang - timedelta(minutes=menit_lalu)).strftime("%Y-%m-%d %H:%M:%S")

def kirim_batch(nama_file, baris, tunggu=8):
    df = pd.DataFrame(baris, columns=kolom)
    df.to_csv(f"{folder_stream}/{nama_file}.csv", index=False)
    semua_data.append(df)
    print(f"\n=== {nama_file} dikirim ({len(df)} baris). Menunggu Spark memproses... ===")
    time.sleep(tunggu)

    print("Hasil Spark (agregasi_cabang):")
    spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

    gabungan = pd.concat(semua_data)
    gabungan["total_harga"] = gabungan["qty"] * gabungan["harga_satuan"]
    cek = gabungan.groupby("nama_cabang").agg(
        total_harga=("total_harga", "sum"),
        jumlah_pesanan=("pesanan_id", "count")
    ).reset_index()
    print("Verifikasi manual (pandas):")
    print(cek.to_string(index=False))

In [ ]:
# Batch1
kirim_batch("batch_1", [
    ("P1", "Magelang",   "Kopi Susu", 2, 18000, t(0)),
    ("P2", "Yogyakarta", "Americano", 1, 20000, t(0)),
    ("P3", "Semarang",   "Latte",     3, 25000, t(0)),
], tunggu=10)

In [ ]:
# Batch 2
kirim_batch("batch_2", [
    ("P4", "Magelang", "Cappuccino", 2, 22000, t(3)),
    ("P5", "Solo",     "Kopi Susu",  4, 18000, t(3)),
    ("P6", "Magelang", "Americano",  1, 20000, t(3)),
])

In [ ]:
# Batch 3
kirim_batch("batch_3", [
    ("P7",  "Yogyakarta", "Latte",      2, 25000, t(6)),
    ("P8",  "Semarang",   "Kopi Susu",  5, 18000, t(6)),
    ("P9",  "Solo",       "Cappuccino", 1, 22000, t(6)),
    ("P10", "Magelang",   "Latte",      2, 25000, t(6)),
])

In [ ]:
# Hasil window 2 menit (Bagian C)
print("\n=== Hasil agregasi_waktu ===")
spark.sql("""
    SELECT window.start, window.end, total_harga
    FROM agregasi_waktu
    ORDER BY window.start
""").show(truncate=False)

In [ ]:
#E. Sink Berbasis Berkas
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_ckpt) \
    .outputMode("append") \
    .start()

# Kirim satu batch baru
pd.DataFrame({
    "pesanan_id": ["P11", "P12"],
    "nama_cabang": ["Solo", "Semarang"],
    "nama_menu": ["Americano", "Latte"],
    "qty": [3, 1],
    "harga_satuan": [20000, 25000],
    "waktu_pesan": [t(0), t(0)],
}).to_csv(f"{folder_stream}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark menulis Parquet...")
time.sleep(10)

# Bukti berkas Parquet tersimpan
!ls tugas10_output

# Baca kembali sebagai DataFrame batch biasa
df_hasil = spark.read.parquet(folder_output)
print("Jumlah baris di Parquet:", df_hasil.count())
df_hasil.orderBy("pesanan_id").show(truncate=False)

In [ ]:
# Tutup semua
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")